In [2]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler

from sklearn.cross_decomposition import PLSRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor,GradientBoostingRegressor
from sklearn.linear_model import Lasso, ElasticNet
from sklearn.neural_network import MLPRegressor

from xgboost import XGBRegressor


In [3]:

# Download all kinases from API

url = "https://kinepik.org/api/0/kinases/all"

response = requests.get(url)

# Convert API response to DataFrame
kinase_df = pd.DataFrame(response.json())

# Keep only UniProt ID and Gene Symbol
kinase_df = pd.DataFrame({
    "UniprotID": kinase_df["UniprotID"],
    "GeneName": kinase_df["GeneInfo"].apply(lambda x: x["MappedGene"])
})

# Display table
kinase_df.head()

,UniprotID,GeneName
0,P06239,LCK
1,P12931,SRC
2,P06241,FYN
3,P00519,ABL1
4,P24941,CDK2


In [4]:
print("Total Kinases:", len(kinase_df))

Total Kinases: 504


In [5]:
# get phosphosites for all kinases

def get_phosphosites(kinase_id):

    url = ( "https://kinepik.org/api/0/kinases/specific?"
        f"kinase_ids={kinase_id}&phosphosites=sites"
    )

    response = requests.get(url)
    data = response.json()

    if len(data) ==0:
        return []

    return data[0]["PhosphositesOnKinase"]

In [6]:

# Collect phosphosites for all 504 kinases

all_phosphosites = []

for _, row in kinase_df.iterrows():

    kinase_id = row["UniprotID"]
    gene = row["GeneName"]

    sites = get_phosphosites(kinase_id)

    all_phosphosites.append({
        "UniprotID": kinase_id,
        "GeneName": gene,
        "TotalPhosphosites": len(sites),
        "Phosphosites": sites
    })

phosphosite_df = pd.DataFrame(all_phosphosites)

phosphosite_df.head(10)

,UniprotID,GeneName,TotalPhosphosites,Phosphosites
0,P06239,LCK,6,"[LCK(Y394), LCK(Y505), LCK(Y192), LCK(S42), LC..."
1,P12931,SRC,17,"[SRC(Y216), SRC(Y338), SRC(Y419), SRC(Y530), S..."
2,P06241,FYN,13,"[FYN(Y39), FYN(Y420), FYN(Y28), FYN(Y30), FYN(..."
3,P00519,ABL1,37,"[ABL1(S446), ABL1(S465), ABL1(Y393), ABL1(Y226..."
4,P24941,CDK2,9,"[CDK2(T160), CDK2(Y168), CDK2(S46), CDK2(T165)..."
5,O14757,CHEK1,16,"[CHEK1(S301), CHEK1(S286), CHEK1(S345), CHEK1(..."
6,Q96GD4,AURKB,8,"[AURKB(T232), AURKB(T16), AURKB(S7), AURKB(S33..."
7,P06493,CDK1,12,"[CDK1(S39), CDK1(T161), CDK1(T222), CDK1(Y15),..."
8,O15530,PDPK1,26,"[PDPK1(T513), PDPK1(S241), PDPK1(S393), PDPK1(..."
9,P07949,RET,16,"[RET(Y809), RET(Y1090), RET(Y826), RET(Y1029),..."


In [7]:
# Save phosphosite counts

phosphosite_df.to_csv(
    "phosphosite_counts.csv",
    index=False
)

print("Saved phosphosite_counts.csv")

Saved phosphosite_counts.csv


In [8]:
# get fc values for one phosphosite

def get_fc(phosphosite):

    url = ( "https://kinepik.org/api/0/perturbation/fc?"
        f"type=target_phosphosite&id={phosphosite}"
        "&cell_line=HL60&confidence=1"
    )

    response = requests.get(url)

    return response.json()

In [ ]:
# Download FC data for all phosphosites

fc_rows = []

for _, row in phosphosite_df.iterrows():

    kinase_id = row["UniprotID"]
    gene = row["GeneName"]

    for site in row["Phosphosites"]:

        try:
            fc_data = get_fc(site)

            for record in fc_data:

                info = record[site]

                fc_rows.append({
                    "UniprotID": kinase_id,
                    "GeneName": gene,
                    "Phosphosite": site,
                    "Perturbation": info["Perturbation"],
                    "CellLine": info["CellLine"],
                    "FC": float(info["FC"])   # <-- FIXED
                })

        except:
            continue

fc_df = pd.DataFrame(fc_rows)

In [ ]:
fc_df.to_csv(
    "fc_HL60.csv",
    index=False
)

print("Saved fc_HL60.csv")

Saved fc_HL60.csv


In [ ]:
print("Number of FC rows:", len(fc_rows))

Number of FC rows: 46055


In [ ]:
print(fc_df.shape)

fc_df.head()

fc_df.columns

fc_df.info()

(46055, 6)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46055 entries, 0 to 46054
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   UniprotID     46055 non-null  object 
 1   GeneName      46055 non-null  object 
 2   Phosphosite   46055 non-null  object 
 3   Perturbation  46055 non-null  object 
 4   CellLine      46055 non-null  object 
 5   FC            46055 non-null  float64
dtypes: float64(1), object(5)
memory usage: 2.1+ MB


In [ ]:
ksea_df = pd.read_csv("ksea_HL60.csv")

In [ ]:
ksea_df.shape

ksea_df.head()

ksea_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13725 entries, 0 to 13724
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   UniprotID     13725 non-null  object 
 1   GeneName      13725 non-null  object 
 2   Perturbation  13725 non-null  object 
 3   CellLine      13725 non-null  object 
 4   KSEA_z_score  10370 non-null  float64
dtypes: float64(1), object(4)
memory usage: 536.3+ KB


In [ ]:
fc_df.shape

(46055, 6)

In [ ]:
ksea_df.shape

(13725, 5)

In [ ]:
merged_df = pd.merge(
    fc_df,
    ksea_df,
    on=["UniprotID", "GeneName", "Perturbation", "CellLine"],
    how="inner"
)

print(merged_df.shape)
merged_df.head()

(46055, 7)


,UniprotID,GeneName,Phosphosite,Perturbation,CellLine,FC,KSEA_z_score
0,P12931,SRC,SRC(S75),AZD5438,HL60,-0.526308,0.349678
1,P12931,SRC,SRC(S75),FRAX486,HL60,-0.722243,-0.062325
2,P12931,SRC,SRC(S75),Ku0063794,HL60,-0.460889,1.004740
3,P12931,SRC,SRC(S75),PD153035,HL60,-0.919130,0.415577
4,P12931,SRC,SRC(S75),PF3758309,HL60,-0.207614,-0.236652


In [ ]:
# Remove rows with missing KSEA

usable_df = merged_df.dropna(subset=["KSEA_z_score"]).copy()

print(usable_df.shape)

(35685, 7)


In [ ]:
#Count phosphosites per kinase

phosphosite_count = (
    usable_df
    .groupby("UniprotID")["Phosphosite"]
    .nunique()
    .reset_index()

)

phosphosite_count.columns = ["UniprotID", "Num_Phosphosites"]

phosphosite_count

,UniprotID,Num_Phosphosites
0,O00418,5
1,O00506,1
2,O14578,6
3,O14733,2
4,O14757,2
...,...,...
165,Q9Y3S1,4
166,Q9Y463,1
167,Q9Y4K4,2
168,Q9Y572,2


In [ ]:
#keep kinases with at least 3 phosphosites

usable_kinases = phosphosite_count[
    phosphosite_count["Num_Phosphosites"] >= 3
]

print(usable_kinases.shape)

usable_kinases

(84, 2)


,UniprotID,Num_Phosphosites
0,O00418,5
2,O14578,6
7,O15075,5
11,O43318,4
12,O43353,4
...,...,...
155,Q9NYV4,22
161,Q9UKE5,6
163,Q9Y2K2,8
164,Q9Y2U5,8


In [ ]:
usable_kinases.info()

<class 'pandas.core.frame.DataFrame'>
Index: 84 entries, 0 to 165
Data columns (total 2 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   UniprotID         84 non-null     object
 1   Num_Phosphosites  84 non-null     int64 
dtypes: int64(1), object(1)
memory usage: 2.0+ KB


In [ ]:
# store one feature matrix for each kinase
feature_matrices = {}

In [ ]:
#Build one matrix per kinase

for kinase_id in usable_kinases["UniprotID"]:
    
    #Get data for one kinase
    kinase_data = usable_df[
        usable_df["UniprotID"] ==kinase_id 
    ].copy()

    # Convert long format to wide format
    kinase_matrix = kinase_data.pivot_table(
        index="Perturbation",
        columns="Phosphosite",
        values="FC",
        aggfunc="first" 
    )

    #Get one KSEA value for each perturbation
    ksea = (
        kinase_data[
            ["Perturbation","KSEA_z_score"] 
        ]
        .drop_duplicates()
        .set_index("Perturbation") 
    )

    # Join FC features with KSEA target
    kinase_matrix = kinase_matrix.join(ksea)

    #Store using Uniprot ID
    feature_matrices[kinase_id] = kinase_matrix

print(f"Created {len(feature_matrices)} feature matrices.")

Created 84 feature matrices.


In [ ]:
print(len(feature_matrices))

84


In [ ]:
# Let's inspect one matrix

first_kinase = list(feature_matrices.keys())[0]

print(first_kinase)

feature_matrices[first_kinase]

O00418


,EEF2K(S18),EEF2K(S66),EEF2K(S72),EEF2K(S74),EEF2K(Y69),KSEA_z_score
Perturbation,,,,,,
AC220,-0.376882,-0.000821,-1.518717,-1.384875,-1.537695,1.052219
AT13148,0.329114,0.311671,0.003186,-0.039981,0.013190,0.229816
AZ20,0.076842,0.494227,-1.833126,-1.381381,-1.609283,0.542754
AZD1480,0.054225,-1.842059,0.912331,0.653138,0.954375,-0.447402
AZD3759,-0.465411,-0.997637,-0.926101,-1.060431,-1.973416,-1.159261
...,...,...,...,...,...,...
Torin,-0.009611,-1.425666,-2.092644,-3.072661,-0.608449,1.024159
Trametinib,0.192274,-0.845325,-0.355835,-0.304800,-0.597217,1.044033
U73122,0.172612,-1.654911,-1.704219,-1.028978,-1.421159,-0.158462


In [ ]:
# Create dictionaries for x and y

# Store features and targets separately
X_data = {}
y_data = {}

In [ ]:
# Split every kinase matrix

for kinase_id, matrix in feature_matrices.items():

    # Remove rows where target is missing
    matrix = matrix.dropna(subset=["KSEA_z_score"])

    # Features (all phosphosite FC values)
    X = matrix.drop(columns=["KSEA_z_score"])

    # Target (kinase activity)
    y = matrix["KSEA_z_score"]

    # Store
    X_data[kinase_id] = X
    y_data[kinase_id] = y

print(f"Prepared X and y for {len(X_data)} kinases.")

Prepared X and y for 84 kinases.


In [ ]:
# Inspect one kinase

first_kinase = list(X_data.keys())[0]

print("Kinase:", first_kinase)

print("\nX shape:", X_data[first_kinase].shape)
print("y shape:", y_data[first_kinase].shape)

X_data[first_kinase].head()

Kinase: O00418

X shape: (61, 5)
y shape: (61,)


,EEF2K(S18),EEF2K(S66),EEF2K(S72),EEF2K(S74),EEF2K(Y69)
Perturbation,,,,,
AC220,-0.376882,-0.000821,-1.518717,-1.384875,-1.537695
AT13148,0.329114,0.311671,0.003186,-0.039981,0.013190
AZ20,0.076842,0.494227,-1.833126,-1.381381,-1.609283
AZD1480,0.054225,-1.842059,0.912331,0.653138,0.954375
AZD3759,-0.465411,-0.997637,-0.926101,-1.060431,-1.973416


In [ ]:
# count missing values

missing_summary = {}

for kinase_id, X in X_data.items():
    
    missing_summary[kinase_id] = X.isna().sum().sum()

missing_df = (
    pd.DataFrame.from_dict(
        missing_summary,
        orient="index",
        columns=["Missing_FC_Values"] 
    )
    .reset_index()

)

missing_df.columns = ["UniprotID","Missing_FC_Values"]

missing_df.sort_values(
    by="Missing_FC_Values",
    ascending=False,
    inplace=True 
)

missing_df.head(10)

In [ ]:
missing_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 84 entries, 0 to 83
Data columns (total 2 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   UniprotID          84 non-null     object
 1   Missing_FC_Values  84 non-null     int64 
dtypes: int64(1), object(1)
memory usage: 2.0+ KB


In [ ]:
print(missing_df.shape)

(84, 2)


In [ ]:
#Feature scaling

X_scaled ={}

for kinase_id in X_data:

    scaler = StandardScaler()

    X_scaled[kinase_id] = pd.DataFrame(
        scaler.fit_transform(X_data[kinase_id]),
        columns=X_data[kinase_id].columns,
        index=X_data[kinase_id].index
    )

print(f"Scaled {len(X_scaled)} kinase feature matrices.")

Scaled 84 kinase feature matrices.


In [ ]:
#inspect one kinase

first_kinase = list(X_scaled.keys())[0]

print(first_kinase)

X_scaled[first_kinase].head()

O00418


,EEF2K(S18),EEF2K(S66),EEF2K(S72),EEF2K(S74),EEF2K(Y69)
Perturbation,,,,,
AC220,-1.016349,0.731289,-0.381109,-0.503105,-0.574835
AT13148,1.555049,1.089451,0.517478,0.482780,0.452950
AZ20,0.636218,1.298686,-0.566748,-0.500544,-0.622277
AZD1480,0.553839,-1.379043,1.054270,0.990877,1.076682
AZD3759,-1.338790,-0.411211,-0.031207,-0.265269,-0.863592


In [ ]:
#Train/test split

X_train = {}
X_test = {}
y_train = {}
y_test = {}

for kinase in X_scaled:

    X_train[kinase], X_test[kinase], y_train[kinase], y_test[kinase] = train_test_split(
        X_scaled[kinase],
        y_data[kinase],
        test_size=0.3,
        random_state=42
    )

print(f"Prepared train/test sets for {len(X_train)} kinases.")

Prepared train/test sets for 84 kinases.


In [ ]:
#verify one kinase

first_kinase = list(X_train.keys())[0]

print("Kinase:", first_kinase)

print("X_train:",X_train[first_kinase].shape)
print("X_test:", X_test[first_kinase].shape)

print("y_train:", y_train[first_kinase].shape)
print("y_test:", y_test[first_kinase].shape)

Kinase: O00418
X_train: (42, 5)
X_test: (19, 5)
y_train: (42,)
y_test: (19,)


In [ ]:
def evaluate_model(model, X_train, X_test, y_train, y_test):

    # Cross-validation
    cv_scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=5,
        scoring="r2"
    )

    # Train final model
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

    # Metrics
    r2 = r2_score(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)

    return {
        "cv_mean": cv_scores.mean(),
        "cv_std": cv_scores.std(),
        "test_r2": r2,
        "test_mse": mse,
        "prediction": y_pred
    }

In [ ]:
rf = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

results = evaluate_model(
    rf,
    X_train[first_kinase],
    X_test[first_kinase],
    y_train[first_kinase],
    y_test[first_kinase]
)

print(results)

{'cv_mean': -0.4572055609801276, 'cv_std': 0.321889477595916, 'test_r2': -0.08569533461970047, 'test_mse': 1.9636180704212867, 'prediction': array([-0.32311352,  0.56168875, -0.14906944,  0.13232722,  0.08978867,
       -0.47269079,  0.03885068,  0.02565101,  0.07750123, -0.44402289,
        0.16698557, -0.52014747, -0.58223171, -0.47827309, -0.44008656,
        0.66113153,  0.10056097, -0.40111233, -0.34956809])}


In [ ]:
def get_model(model_name, n_features=None):

    if model_name == "RandomForest":
        return RandomForestRegressor(
            n_estimators=100,
            random_state=42
        )

    elif model_name == "XGBoost":
        return XGBRegressor(
            objective="reg:squarederror",
            n_estimators=100,
            learning_rate=0.1,
            random_state=42
        )

    elif model_name == "PLS":

        n_components = min(3, n_features)

        return PLSRegression(
            n_components=n_components
        )

    elif model_name == "SVR":

        return SVR(
            kernel="linear",
            C=1.0,
            epsilon=0.1
        )

    elif model_name == "Lasso":

        return Lasso(
            alpha=0.1
        )

    elif model_name == "ElasticNet":

        return ElasticNet(
            alpha=0.1,
            l1_ratio=0.5
        )

    elif model_name == "GradientBoosting":

        return GradientBoostingRegressor(
            n_estimators=100,
            learning_rate=0.1,
            random_state=42
        )

    elif model_name == "MLP":

        hidden_size = min(max(n_features - 1, 1), 10)

        return MLPRegressor(
            hidden_layer_sizes=(hidden_size,),
            activation="relu",
            solver="adam",
            max_iter=5000,
            random_state=1
        )

    else:

        raise ValueError(f"Unknown model: {model_name}")

In [ ]:
#Test the function

first_kinase = list(X_train.keys())[0]

print(first_kinase)

n_features = X_train[first_kinase].shape[1]

print(n_features)

O00418
5


In [ ]:
#Build one random forest model
rf = get_model(
    "RandomForest",
    n_features
)

print(rf)

RandomForestRegressor(random_state=42)


In [ ]:
#Build one MLP model
mlp = get_model(
    "MLP",
    n_features
)

print(mlp)

MLPRegressor(hidden_layer_sizes=(4,), max_iter=5000, random_state=1)


In [ ]:
#Build one XGBoost model
xgb = get_model(
    "XGBoost",
    n_features
)

print(xgb)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.1, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=100,
             n_jobs=None, num_parallel_tree=None, ...)


In [ ]:
#Build one PLS model
pls = get_model(
    "PLS",
    n_features
)

print(pls)

PLSRegression(n_components=3)


In [ ]:
#Build one SVR model 
svr = get_model(
    "SVR",
    n_features
)

print(svr)

SVR(kernel='linear')


In [ ]:
#Build one Gradient Boosting model
gbr = get_model(
    "GradientBoosting",
    n_features
)

print(gbr)

GradientBoostingRegressor(random_state=42)


In [ ]:
#Build one Lasso model 
lasso = get_model(
    "Lasso",
    n_features
)

print(lasso)

Lasso(alpha=0.1)


In [ ]:
#Build one ElasticNet model
enet = get_model(
    "ElasticNet",
    n_features
)

print(enet)

ElasticNet(alpha=0.1)


In [ ]:
#Evaluate one model on first kinase
# First kinase
first_kinase = list(X_train.keys())[0]

# Number of phosphosite features
n_features = X_train[first_kinase].shape[1]

# Choose model
model = get_model(
    "RandomForest",
    n_features
)

# Evaluate model
results = evaluate_model(
    model,
    X_train[first_kinase],
    X_test[first_kinase],
    y_train[first_kinase],
    y_test[first_kinase]
)

results


{'cv_mean': -0.4572055609801276,
 'cv_std': 0.321889477595916,
 'test_r2': -0.08569533461970047,
 'test_mse': 1.9636180704212867,
 'prediction': array([-0.32311352,  0.56168875, -0.14906944,  0.13232722,  0.08978867,
        -0.47269079,  0.03885068,  0.02565101,  0.07750123, -0.44402289,
         0.16698557, -0.52014747, -0.58223171, -0.47827309, -0.44008656,
         0.66113153,  0.10056097, -0.40111233, -0.34956809])}

In [ ]:
results_df = pd.DataFrame({
    "Metric": [
        "CV Mean R²",
        "CV Std R²",
        "Test R²",
        "Test MSE"
    ],
    "Value": [
        results["cv_mean"],
        results["cv_std"],
        results["test_r2"],
        results["test_mse"]
    ]
})

results_df

,Metric,Value
0,CV Mean R²,-0.457206
1,CV Std R²,0.321889
2,Test R²,-0.085695
3,Test MSE,1.963618


In [ ]:
#comparing all 8 models automatically on the first kinase
#create the list of models 
model_names = [
    "RandomForest",
    "XGBoost",
    "PLS",
    "SVR",
    "Lasso",
    "ElasticNet",
    "GradientBoosting",
    "MLP"
]

In [ ]:
#Create an empty list

comparison_results =[]

In [ ]:
#Loop through every model
for model_name in model_names:

    model = get_model(
        model_name,
        n_features
    )

    results = evaluate_model(
        model,
        X_train[first_kinase],
        X_test[first_kinase],
        y_train[first_kinase],
        y_test[first_kinase]
    )

    comparison_results.append({
        "Model": model_name,
        "CV Mean R²": results["cv_mean"],
        "CV Std R²": results["cv_std"],
        "Test R²": results["test_r2"],
        "Test MSE": results["test_mse"]
    })

In [ ]:
#Convert to a DataFrame

comparison_df = pd.DataFrame(comparison_results)

comparison_df

,Model,CV Mean R²,CV Std R²,Test R²,Test MSE
0,RandomForest,-0.457206,0.321889,-0.085695,1.963618
1,XGBoost,-1.711004,1.304069,-0.365468,2.469622
2,PLS,-0.373195,0.323411,0.099489,1.628690
3,SVR,-0.395335,0.428539,0.119233,1.592978
4,Lasso,-0.197124,0.162209,0.051979,1.714617
5,ElasticNet,-0.212141,0.183806,0.093086,1.640269
6,GradientBoosting,-1.613462,0.648908,-0.316819,2.381634
7,MLP,-0.996252,1.086327,0.178063,1.486578


In [ ]:
#Sort from best to worst
comparison_df = comparison_df.sort_values(
    by="Test R²",
    ascending=False
)

comparison_df

,Model,CV Mean R²,CV Std R²,Test R²,Test MSE
7,MLP,-0.996252,1.086327,0.178063,1.486578
3,SVR,-0.395335,0.428539,0.119233,1.592978
2,PLS,-0.373195,0.323411,0.099489,1.628690
5,ElasticNet,-0.212141,0.183806,0.093086,1.640269
4,Lasso,-0.197124,0.162209,0.051979,1.714617
0,RandomForest,-0.457206,0.321889,-0.085695,1.963618
6,GradientBoosting,-1.613462,0.648908,-0.316819,2.381634
1,XGBoost,-1.711004,1.304069,-0.365468,2.469622


In [ ]:
#Create an empty results Dataframe

all_results = pd.DataFrame(columns=[
    "Kinase",
    "Model",
    "CV Mean R²",
    "CV Std R²",
    "Test R²",
    "Test MSE"
])

In [ ]:
#Create the list of model names

model_names = [
    "RandomForest",
    "XGBoost",
    "PLS",
    "SVR",
    "Lasso",
    "ElasticNet",
    "GradientBoosting",
    "MLP"
]

In [ ]:
gene_lookup = (
    usable_df[["UniprotID", "GeneName"]]
    .drop_duplicates()
    .set_index("UniprotID")["GeneName"]
    .to_dict()
)

In [ ]:
prediction_results = []

In [ ]:
#Loop through all 85 kinases and all 8 models

for kinase in X_train.keys():

    print(f"\nProcessing kinase: {kinase}")

    # Number of phosphosite features for this kinase
    n_features = X_train[kinase].shape[1]

    # Loop through all models
    for model_name in model_names:

        print(f"   Running {model_name}...")

        # Create a fresh model
        model = get_model(model_name, n_features)

        # Evaluate the model
        results = evaluate_model(
            model,
            X_train[kinase],
            X_test[kinase],
            y_train[kinase],
            y_test[kinase]
        )

        for actual, predicted in zip(y_test[kinase],results["prediction"]):
            
            prediction_results.append({
                "Kinase": kinase,
                "GeneName": gene_lookup.get(kinase, "Unkown"),
                "Model": model_name,
                "Actual": actual,
                "Predicted": predicted 
            })

        # Save results
        all_results.loc[len(all_results)] = [
            kinase,
            model_name,
            results["cv_mean"],
            results["cv_std"],
            results["test_r2"],
            results["test_mse"]
        ]


Processing kinase: O00418
   Running RandomForest...
   Running XGBoost...
   Running PLS...
   Running SVR...
   Running Lasso...
   Running ElasticNet...
   Running GradientBoosting...
   Running MLP...

Processing kinase: O14578
   Running RandomForest...
   Running XGBoost...
   Running PLS...
   Running SVR...
   Running Lasso...
   Running ElasticNet...
   Running GradientBoosting...
   Running MLP...

Processing kinase: O15075
   Running RandomForest...
   Running XGBoost...
   Running PLS...
   Running SVR...
   Running Lasso...
   Running ElasticNet...
   Running GradientBoosting...
   Running MLP...

Processing kinase: O43318
   Running RandomForest...
   Running XGBoost...
   Running PLS...
   Running SVR...
   Running Lasso...
   Running ElasticNet...
   Running GradientBoosting...
   Running MLP...

Processing kinase: O43353
   Running RandomForest...
   Running XGBoost...
   Running PLS...
   Running SVR...
   Running Lasso...
   Running ElasticNet...
   Running Gradient

c:\Users\Sofie\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (5000) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\Sofie\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (5000) reached and the optimization hasn't converged yet.
  warnings.warn(



Processing kinase: Q8IWQ3
   Running RandomForest...
   Running XGBoost...
   Running PLS...
   Running SVR...
   Running Lasso...
   Running ElasticNet...
   Running GradientBoosting...
   Running MLP...

Processing kinase: Q8TD19
   Running RandomForest...
   Running XGBoost...
   Running PLS...
   Running SVR...
   Running Lasso...
   Running ElasticNet...
   Running GradientBoosting...
   Running MLP...

Processing kinase: Q92630
   Running RandomForest...
   Running XGBoost...
   Running PLS...
   Running SVR...
   Running Lasso...
   Running ElasticNet...
   Running GradientBoosting...
   Running MLP...

Processing kinase: Q96GX5
   Running RandomForest...
   Running XGBoost...
   Running PLS...
   Running SVR...
   Running Lasso...
   Running ElasticNet...
   Running GradientBoosting...
   Running MLP...

Processing kinase: Q96Q15
   Running RandomForest...
   Running XGBoost...
   Running PLS...
   Running SVR...
   Running Lasso...
   Running ElasticNet...
   Running Gradient

In [ ]:
prediction_df = pd.DataFrame(prediction_results)

prediction_df.to_csv(
    "all_predictions.csv",
    index=False
)

print(prediction_df.shape)

prediction_df.head()

(12768, 5)


,Kinase,GeneName,Model,Actual,Predicted
0,O00418,EEF2K,RandomForest,1.052219,-0.323114
1,O00418,EEF2K,RandomForest,-1.198411,0.561689
2,O00418,EEF2K,RandomForest,-5.056908,-0.149069
3,O00418,EEF2K,RandomForest,0.538401,0.132327
4,O00418,EEF2K,RandomForest,-0.508058,0.089789


In [ ]:
all_results.shape

(672, 6)

In [ ]:
all_results

,Kinase,Model,CV Mean R²,CV Std R²,Test R²,Test MSE
0,O00418,RandomForest,-0.457206,0.321889,-0.085695,1.963618
1,O00418,XGBoost,-1.711004,1.304069,-0.365468,2.469622
2,O00418,PLS,-0.373195,0.323411,0.099489,1.628690
3,O00418,SVR,-0.395335,0.428539,0.119233,1.592978
4,O00418,Lasso,-0.197124,0.162209,0.051979,1.714617
...,...,...,...,...,...,...
667,Q9Y3S1,SVR,-0.136534,0.120258,0.068875,0.755510
668,Q9Y3S1,Lasso,-0.234592,0.161102,0.058289,0.764099
669,Q9Y3S1,ElasticNet,-0.243162,0.191590,0.095069,0.734256
670,Q9Y3S1,GradientBoosting,-0.714520,0.641553,-0.396299,1.132949


In [ ]:

all_results.to_csv(
    "all_models_85_kinases.csv",
    index=False
)

print("Saved!")

Saved!


In [ ]:
#which model wins for each kinase

best_models = (
    all_results
    .sort_values("Test R²", ascending=False)
    .groupby("Kinase")
    .first()
    .reset_index()
)

best_models

,Kinase,Model,CV Mean R²,CV Std R²,Test R²,Test MSE
0,O00418,MLP,-9.962515e-01,1.086327e+00,0.178063,1.486578
1,O14578,Lasso,2.085839e-01,1.911244e-01,0.297690,0.371181
2,O15075,Lasso,-5.141529e-02,5.501650e-02,-0.042251,0.222621
3,O43318,SVR,-7.480406e-01,5.382230e-01,0.101737,0.585815
4,O43353,MLP,-5.459856e-01,8.595484e-01,0.319197,0.440717
...,...,...,...,...,...,...
79,Q9NYV4,PLS,-6.472238e-01,4.908689e-01,0.274259,0.887720
80,Q9UKE5,SVR,9.674785e-01,1.893120e-02,0.983224,0.008478
81,Q9Y2K2,RandomForest,-6.036395e-02,6.327931e-01,0.582937,0.351974
82,Q9Y2U5,SVR,-8.965497e-01,1.325927e+00,0.133306,1.860296


In [ ]:
#count how many kinases each model wins

best_models["Model"].value_counts()

Model
Lasso               19
SVR                 15
MLP                 14
RandomForest        10
PLS                  9
XGBoost              6
GradientBoosting     6
ElasticNet           5
Name: count, dtype: int64

In [ ]:
best_models.to_csv(
    "best_model_per_kinase.csv",
    index=False
)